## 정보요약

- 여러 요약 방식들을 하나의 Python 파일에서 동시에 실행 및 성능 비교
- 각 방법을 순차적으로 실행하고, 동일한 입력 문서에 대해 결과를 나란히 출력합니다

    - Stuff Summary
    - Map Reduce Summary
    - Map Refine
    - Chain of Density (COD)
    - Clustering Map Refined


In [ ]:
# Stuff Summary
"""
그냥 AI모델에게 요청하기 + 템플릿/요청사항 등을 함께 요청
- 장점: 가장 간단, 짧게 제일 중요한것만 짚어낼 가능성 높음
- 한계 : 일반적으로 4096토큰, 128K모델 사용해도 원문의 토큰이 더 많을 경우 짤린 정보에대해서만 요약. 문서중 가장 중요하다고 여겨지는 정보에대해서만 요약됨(정확하더라도 내용이 부족할 수 있음).
"""
# Map Reduce Summary
"""
배경: Stuff방식은 결과물의 밀도가 떨어질 수 있음.
장점: Stuff방식보다 효율적으로 요약
방법: 긴 문서를 Chunk단위로 쪼개서 요약 (보통은 Page단위로 약 2k토큰)
"""
# Map Refine
"""
방법: CHUNK단위로 나누는건 동일 + 앞의 요약과 합쳐서 점진적으로 내용을 보강하는 프로세스 
"""
# Chain of Density (COD)
"""
참조논문 : From Sparse to Dense 
- 점진적 개선 : 짧은 요약 생성 후 길이를 늘리지 않으면서 누락된 중요 개체들을 반복적으로 통합하는 프로세스.
- 성과 : 처음에는 압도적으로 부족하지만, 점진적으로 다섯번 정도면 일반적인 사람의 퀄리티를 넘어서게됨

**입력 파라미터 설명**

- `content_category`: 콘텐츠 정류(예: 기사, 동영상 녹취록, 블로그 게시물, 연구 논문). 기본값: Article

- `content`: 요약할 콘텐츠

- `entity_range`: 콘텐츠에서 선택하여 요약에 추가할 엔티티의 수의 범위. 기본값은 `1-3`

- `max_words`: 1번 요약시, 요약에 포함할 최대 단어. 기본값은 **80** 입니다.

- `iterations`: 엔티티 고밀도화 라운드 수. 총 요약은 **반복 횟수+1** 입니다. 80단어의 경우 3회 반복이 이상적입니다. 요약이 더 길면 4~5회, 그리고 `entity_range` 를 예를 들어 1~4로 변경하는 것도 도움이 될 수 있습니다. 기본값: 3.

참조링크 : https://arxiv.rog/pdf/2309.04269

POC / 기술블로그 / 코치님블로그 요청 / 
cursor AI -> README, Before After

"""


# Clustering Map Refined
"""
gkamradt 은 긴 문서의 요약에 대해서 흥미로운 제안을 하였습니다.
(Sementic Chunking 만든사람 - in TextSpliter)


* 배경
1. map-reduce 나 map-refine 방식은 모두 시간이 오래 걸리고, 비용이 많이 듬.
2. 따라서, 문서를 몇 개(N 개)의 클러스터로 나눈 뒤, 가장 중심축에서 가까운 문서를 클러스터의 대표 문서로 인지하고, 이를 map-reduce(혹은 map-refine) 방식으로 요약하는 방식을 제안.
 
* 방법
- 정보를 clustering해서 그 군집화된 문서의 핵심 정보(또는 문서)를 선별하여 요약

[출처 - gkamradt](https://github.com/gkamradt/langchain-tutorials/blob/main/data_generation/5%20Levels%20Of%20Summarization%20-%20Novice%20To%20Expert.ipynb)
"""


In [16]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain import hub
from langchain_core.output_parsers import StrOutputParser, SimpleJsonOutputParser
from langchain_teddynote.callbacks import StreamingCallback
from langchain_core.runnables import chain

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.cluster import KMeans
from langchain_upstage import UpstageEmbeddings
import numpy as np

import tempfile
from langchain.document_loaders import PyPDFLoader
import requests



In [17]:
from dotenv import load_dotenv
import os


# API 키 로드
load_dotenv('../../.env')
api_key = os.getenv("OPENAI_API_KEY")


# 모델 초기화
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0, openai_api_key=api_key)
streaming_llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0, streaming=True, callbacks=[StreamingCallback()])


In [19]:

# 문서 로드 함수 수정 (웹에서 직접 다운로드)
def load_docs_from_url(url: str):
    # PDF 임시 다운로드
    response = requests.get(url)
    response.raise_for_status()  # 오류 시 예외 발생

    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp_file:
        tmp_file.write(response.content)
        tmp_pdf_path = tmp_file.name

    # PDF 로드
    loader = PyPDFLoader(tmp_pdf_path)
    docs = loader.load()
    return docs[3:8]  # 실험용 일부 페이지만 사용

# 대상 논문 URL
arxiv_url = "https://arxiv.org/pdf/2309.04269"

# 사용 예시
docs = load_docs_from_url(arxiv_url)


In [20]:



# 문서 로드 및 전처리
def load_docs(path: str):
    loader = PyPDFLoader(path)
    docs = loader.load()
    return docs[3:8]  # 실험용으로 일부만 사용


# --- 요약 방법 1: Stuff ---
def summarize_stuff(docs):
    prompt = hub.pull("teddynote/summary-stuff-documents-korean")
    chain = prompt | llm
    return chain.invoke({"context": docs})

# --- 요약 방법 2: Map-Reduce ---
@chain
def summarize_map_reduce(docs):
    map_prompt = hub.pull("teddynote/map-prompt")
    reduce_prompt = hub.pull("teddynote/reduce-prompt")
    map_chain = map_prompt | llm | StrOutputParser()
    doc_summaries = map_chain.batch(docs)
    reduce_chain = reduce_prompt | streaming_llm | StrOutputParser()
    return reduce_chain.invoke({"doc_summaries": "\n".join(doc_summaries), "language": "Korean"})

# --- 요약 방법 3: Map-Refine ---
@chain
def summarize_map_refine(docs):
    map_prompt = hub.pull("teddynote/map-summary-prompt")
    map_chain = map_prompt | llm | StrOutputParser()
    input_doc = [{"documents": doc.page_content, "language": "Korean"} for doc in docs]
    doc_summaries = map_chain.batch(input_doc)

    refine_prompt = hub.pull("teddynote/refine-prompt")
    refine_chain = refine_prompt | streaming_llm | StrOutputParser()

    previous_summary = doc_summaries[0]
    for current_summary in doc_summaries[1:]:
        previous_summary = refine_chain.invoke({
            "previous_summary": previous_summary,
            "current_summary": current_summary,
            "language": "Korean"
        })
    return previous_summary

# --- 요약 방법 4: Chain of Density ---
def summarize_chain_of_density(docs):
    content = docs[0].page_content  # 하나의 문서만 테스트
    cod_prompt = hub.pull("teddynote/chain-of-density-prompt")
    cod_chain = (
        {
            "content": content,
            "content_category": "Article",
            "entity_range": "1-3",
            "max_words": 80,
            "iterations": 3,
        }
        | cod_prompt
        | llm
        | SimpleJsonOutputParser()
    )
    results = cod_chain.invoke({})
    return results[-1].get("denser_summary", "요약 없음")


# --- 요약 방법 5: Clustering-Map-Refine ---
@chain
def summarize_clustering_map_refine(docs):
    # 전체 텍스트 결합
    texts = "\n\n".join([doc.page_content for doc in docs])
    
    # Chunk 분할
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    split_texts = text_splitter.split_text(texts)

    # 임베딩 (Upstage 또는 OpenAI Embeddings 중 택 1)
    embeddings = UpstageEmbeddings(model="solar-embedding-1-large-passage")
    vectors = embeddings.embed_documents(split_texts)

    # 클러스터링
    num_clusters = 10
    kmeans = KMeans(n_clusters=num_clusters, random_state=123)
    kmeans.fit(vectors)

    # 클러스터 중심과 가장 가까운 문서 선택
    closest_indices = [
        np.argmin(np.linalg.norm(vectors - kmeans.cluster_centers_[i], axis=1))
        for i in range(num_clusters)
    ]

    # 오름차순 정렬로 순서 유지
    selected_indices = sorted(closest_indices)

    # Document 객체 리스트 생성
    selected_docs = [Document(page_content=split_texts[i]) for i in selected_indices]

    # 기존 map-refine chain 재사용
    return summarize_map_refine.invoke(selected_docs)




In [21]:
    

# 실행 및 비교 출력

print("====== Stuff 요약 ======")
print(summarize_stuff(docs))

print("\n====== Map-Reduce 요약 ======")
print(summarize_map_reduce.invoke(docs))

print("\n====== Map-Refine 요약 ======")
print(summarize_map_refine.invoke(docs))

print("\n====== Chain of Density 요약 ======")
print(summarize_chain_of_density(docs))


print("\n====== Clustering-Map-Refine 요약 ======")
print(summarize_clustering_map_refine.invoke(docs))



====== Stuff 요약 ======


/home/jarvis/.local/lib/python3.12/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


content='- 📊 **GPT-4의 CoD 단계별 요약 평가에서 평균 점수는 4.69로 나타났다.**\n- 🏆 **1단계가 100개의 예시 중 가장 많은 1위 투표를 받았다.**\n- 📈 **3단계 요약의 평균 밀도는 약 0.15로, 인간이 작성한 요약과 유사하다.**\n- 🤖 **GPT-4는 인간의 평가와 잘 일치하며, 일부 작업에서 크라우드 소싱 작업자를 초월할 수 있다.**\n- 📉 **요약의 밀도와 정보성은 상관관계가 있지만, 너무 많은 엔티티는 가독성을 저하시킨다.**\n- 🔍 **중간 단계의 CoD 요약이 정보성과 가독성의 균형을 잘 맞춘다.**\n- 📚 **연구 결과는 뉴스 요약에 국한되며, 고유한 모델 가중치는 공유할 수 없다.**' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 217, 'prompt_tokens': 5152, 'total_tokens': 5369, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_92e0377081', 'id': 'chatcmpl-BacCQpN4FXoM2U76bVAbe12fsprLK', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--014bf1d0-1ee7-44a3-8879-495e06e8ad1c-0' usage_metadata={'input_tokens': 5152,

/home/jarvis/.local/lib/python3.12/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(
/home/jarvis/.local/lib/python3.12/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


- GPT-4로 생성된 요약의 평가 결과, 밀도와 정보성이 상관관계가 있으며, 4단계에서 최대치를 기록함.
- 요약의 일관성과 정보성 간의 균형이 존재하며, 중간 단계의 요약이 이 균형을 가장 잘 달성함.
- 밀도가 높고 정보성이 풍부한 요약이 일반 평가에서 높은 점수를 받는 경향이 있으며, 초기 및 최종 단계는 덜 선호됨.
- 요약에서 선호되는 엔티티 밀도는 약 0.15로, 인간이 작성한 요약과 일치함.
- 자동 측정 지표와 인간의 판단 간의 상관관계가 낮아, 요약의 질과 밀도 간의 관계를 정의하고 정량화하기 위한 추가 연구가 필요함.
- 연구는 요약의 밀도가 인간의 질적 선호에 미치는 영향을 분석하며, 적절한 밀도가 선호되지만, 과도한 엔티티는 가독성과 일관성을 저해함.
- 엔티티 체인을 생성하는 것을 요약 모델의 세밀한 조정을 위한 계획 단계로 제안하며, 키워드나 순수 추출 단위의 사용과 대조됨.
- 고정 길이 및 가변 밀도의 요약 연구를 촉진하기 위해 주석이 달린 평가 데이터셋과 더 큰 비주석 훈련 데이터셋이 공개됨.
- 문서는 텍스트 요약과 관련된 다양한 연구 및 데이터셋을 언급하며, 추출적 및 추상적 요약 기술의 발전을 강조함.
- 대형 언어 모델의 요약 및 평가 맥락에서의 도전과 응용에 대해 논의함.
- 요약 모델 개선에 있어 인간 피드백의 중요성이 강조되며, 특히 GPT-4 및 최근 모델과 관련하여 언급됨.
- 엔티티 중심 요약과 생성된 요약의 사실적 일관성 필요성에 초점을 맞춘 여러 논문이 인용됨.
- 뉴스 요약을 위한 대형 언어 모델의 벤치마킹에 대한 언급이 있으며, 이 분야의 지속적인 연구를 나타냄.
- 요약 평가 기준은 정보성, 질, 일관성, 귀속 가능성 및 일반 선호와 같은 품질 지표에 기반함.
- 인간의 선호와 GPT-4의 평가 점수 간의 피어슨 상관관계가 낮지만, 일반 평가를 포착하기 위해 설계된 프롬프트는 가장 높은 상관관계를 보임.
- 품질 지표의 정의는 이전 요약 주석 작업에서 패러프레이즈됨.- GPT-4로 생성된 요약의 평가 

/home/jarvis/.local/lib/python3.12/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(
/home/jarvis/.local/lib/python3.12/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


- CoD 단계별 요약의 질적 평가에서, 3단계 요약이 가장 선호되는 경향이 있으며, 평균적으로 약 0.15의 엔티티 밀도가 선호됨.
- 요약 밀도화가 인간의 전반적인 품질 선호에 미치는 영향을 연구한 결과, 적정한 밀도가 선호되지만 너무 많은 개체가 포함되면 가독성과 일관성을 유지하기 어려운 것으로 나타남.
- GPT-4는 인간의 평가와 잘 일치하며, 요약의 정보성, 품질, 일관성, 귀속성 및 전반적인 품질을 평가하는 데 효과적임.
- 요약의 정보성과 일관성 간에는 명확한 상충 관계가 존재하며, 중간 단계의 CoD 요약이 이 균형을 가장 잘 달성함.
- CoD 단계 4에서 정보성이 최고 점수를 기록했지만, 품질과 일관성은 더 빨리 감소함.
- 연구 결과를 바탕으로 고정 길이, 가변 밀도 요약에 대한 추가 연구를 위해 주석이 달린 테스트 세트와 더 큰 비주석 훈련 세트를 오픈 소스 형태로 제공함.
- 단일 도메인인 뉴스 요약에 대한 분석만 수행하였으며, 주석 수준의 합의는 높지 않았지만 시스템 수준의 경향은 나타남.
- 향후 연구에서는 요약의 정보성과 가독성 간의 상충 관계를 보다 정밀하게 정의하고 정량화할 필요가 있음.- CoD 단계별 요약의 질적 평가에서, 3단계 요약이 가장 선호되는 경향이 있으며, 평균적으로 약 0.15의 엔티티 밀도가 선호됨.
- 요약 밀도화가 인간의 전반적인 품질 선호에 미치는 영향을 연구한 결과, 적정한 밀도가 선호되지만 너무 많은 개체가 포함되면 가독성과 일관성을 유지하기 어려운 것으로 나타남.
- GPT-4는 인간의 평가와 잘 일치하며, 요약의 정보성, 품질, 일관성, 귀속성 및 전반적인 품질을 평가하는 데 효과적임.
- 요약의 정보성과 일관성 간에는 명확한 상충 관계가 존재하며, 중간 단계의 CoD 요약이 이 균형을 가장 잘 달성함.
- CoD 단계 4에서 정보성이 최고 점수를 기록했지만, 품질과 일관성은 더 빨리 감소함.
- 연구 결과를 바탕으로 고정 길이, 가변 밀도 요약에 대한 추가 연구를 위해 주석이 달린 테스트 세트와

/home/jarvis/.local/lib/python3.12/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


TypeError: Expected a Runnable, callable or dict.Instead got an unsupported type: <class 'str'>